# Problem: Dining Philosophers Problem Using Monitor

Five philosophers are sitting around a circular table.  
Each philosopher has one chopstick on the left side and one chopstick on the right side.

A philosopher can be in one of three states:

1. THINKING
2. HUNGRY
3. EATING

A philosopher can eat only when both left and right chopsticks are available.

Condition:
1. No two neighboring philosophers can eat at the same time.
2. A philosopher must wait if either left or right neighbor is eating.
3. Use Monitor to solve the Dining Philosophers synchronization problem.

Solve this problem using Monitor in Python.

In [1]:
import threading
import time
import random

# --------------------------------------------------
# Dining Philosophers Problem using Monitor
# Following the method shown in the given picture
# --------------------------------------------------

# Number of philosophers
N = 5

# States of philosophers
THINKING = "THINKING"
HUNGRY = "HUNGRY"
EATING = "EATING"


class DiningPhilosophersMonitor:
    def __init__(self):
        # Initially all philosophers are THINKING
        self.state = [THINKING for _ in range(N)]

        # This lock represents the monitor lock
        # Only one philosopher can execute monitor methods at a time
        self.monitor_lock = threading.Lock()

        # Condition variable for each philosopher
        # self_condition[i] is used when philosopher i has to wait
        self.self_condition = [
            threading.Condition(self.monitor_lock) for _ in range(N)
        ]

    def left(self, i):
        # Left neighbor of philosopher i
        # Example: left of philosopher 0 is philosopher 4
        return (i + N - 1) % N

    def right(self, i):
        # Right neighbor of philosopher i
        # Example: right of philosopher 4 is philosopher 0
        return (i + 1) % N

    def test(self, i):
        """
        This function checks whether philosopher i can eat or not.

        According to the picture method:
        Philosopher i can eat only if:
        1. Left neighbor is not EATING
        2. Philosopher i is HUNGRY
        3. Right neighbor is not EATING
        """

        if (
            self.state[self.left(i)] != EATING
            and self.state[i] == HUNGRY
            and self.state[self.right(i)] != EATING
        ):
            # Philosopher i can start eating
            self.state[i] = EATING

            # Wake up philosopher i if he was waiting
            self.self_condition[i].notify()

    def pickup(self, i):
        """
        pickup(i) method from monitor.

        Philosopher i becomes HUNGRY first.
        Then test(i) checks if he can eat.
        If he cannot eat, he waits.
        """

        with self.monitor_lock:
            # Philosopher i is now hungry
            self.state[i] = HUNGRY
            print(f"Philosopher {i} is HUNGRY.")

            # Check whether philosopher i can eat
            self.test(i)

            # If philosopher i cannot eat, he waits
            while self.state[i] != EATING:
                print(f"Philosopher {i} is waiting.")
                self.self_condition[i].wait()

            print(f"Philosopher {i} starts EATING.")

    def putdown(self, i):
        """
        putdown(i) method from monitor.

        Philosopher i finishes eating and becomes THINKING.
        Then he checks whether his left and right neighbors can eat.
        """

        with self.monitor_lock:
            # Philosopher i finishes eating
            self.state[i] = THINKING
            print(f"Philosopher {i} puts down chopsticks and starts THINKING.")

            # Check if left neighbor can eat now
            self.test(self.left(i))

            # Check if right neighbor can eat now
            self.test(self.right(i))


def philosopher(i, monitor):
    """
    Each philosopher repeatedly:
    1. Thinks
    2. Picks up chopsticks
    3. Eats
    4. Puts down chopsticks
    """

    for _ in range(3):
        # Thinking section
        print(f"Philosopher {i} is THINKING.")
        time.sleep(random.randint(1, 3))

        # Try to pick up chopsticks
        monitor.pickup(i)

        # Eating section
        time.sleep(random.randint(1, 2))

        # Put down chopsticks
        monitor.putdown(i)


# Create monitor object
monitor = DiningPhilosophersMonitor()

# Create philosopher threads
philosophers = []

for i in range(N):
    t = threading.Thread(target=philosopher, args=(i, monitor))
    philosophers.append(t)

# Start all philosopher threads
for t in philosophers:
    t.start()

# Wait for all philosopher threads to finish
for t in philosophers:
    t.join()

print("Dining Philosophers problem solved using Monitor.")

Philosopher 0 is THINKING.
Philosopher 1 is THINKING.
Philosopher 2 is THINKING.
Philosopher 3 is THINKING.
Philosopher 4 is THINKING.
Philosopher 2 is HUNGRY.
Philosopher 2 starts EATING.
Philosopher 3 is HUNGRY.
Philosopher 3 is waiting.
Philosopher 1 is HUNGRY.
Philosopher 1 is waiting.
Philosopher 0 is HUNGRY.
Philosopher 0 starts EATING.
Philosopher 4 is HUNGRY.
Philosopher 4 is waiting.
Philosopher 2 puts down chopsticks and starts THINKING.
Philosopher 2 is THINKING.
Philosopher 3 starts EATING.
Philosopher 2 is HUNGRY.
Philosopher 2 is waiting.
Philosopher 3 puts down chopsticks and starts THINKING.
Philosopher 3 is THINKING.
Philosopher 2 starts EATING.
Philosopher 0 puts down chopsticks and starts THINKING.
Philosopher 0 is THINKING.
Philosopher 4 starts EATING.
Philosopher 4 puts down chopsticks and starts THINKING.
Philosopher 4 is THINKING.
Philosopher 3 is HUNGRY.
Philosopher 3 is waiting.
Philosopher 2 puts down chopsticks and starts THINKING.
Philosopher 2 is THINKING.
